# 5.3 — Alineación Semántica Multimodal con CLIP

**Modelo:** `openai/clip-vit-base-patch32`  
**Objetivo:** Evaluar la consistencia semántica entre titulares e imágenes (reales vs. sintéticos) mediante embeddings zero-shot, **sin entrenamiento supervisado**.

---

## Marco Teórico

CLIP (Contrastive Language-Image Pre-training) proyecta imágenes y texto en un **espacio de embeddings compartido** donde la similitud coseno mide la afinidad semántica entre modalidades.

Para cada par $(\text{titular}, \text{imagen})$, extraemos:
- $E_{Img}$ — Embedding visual de la imagen
- $E_{T\_Real}$ — Embedding textual del titular real
- $E_{T\_Sint}$ — Embedding textual del titular sintético (paráfrasis generada por LLM)

### Dos Enfoques Complementarios

| Enfoque | Imagen Evaluada | Hipótesis |
|---|---|---|
| **A — Fidelidad Original** | Imagen Real | $\cos(E_{Img\_Real}, E_{T\_Real}) > \cos(E_{Img\_Real}, E_{T\_Sint})$ → El titular real preserva mayor fidelidad semántica |
| **B — Sesgo de Generación** | Imagen Sintética (FLUX) | $\cos(E_{Img\_Sint}, E_{T\_Sint}) > \cos(E_{Img\_Sint}, E_{T\_Real})$ → *Prompt-Adherence Bias*: la imagen generada se alinea artificialmente con su prompt |

## 1 · Entorno y Dependencias

In [ ]:
# ── Instalación (Colab) ──────────────────────────────────────────────
!pip install -q transformers torch pillow requests tqdm matplotlib seaborn scikit-learn pandas

In [ ]:
import json
import io
import warnings
from pathlib import Path
from collections import defaultdict

import requests
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm
from transformers import CLIPModel, CLIPProcessor

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    accuracy_score,
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.15)

print(f"PyTorch {torch.__version__}")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {DEVICE}")

## 2 · Cargar el Modelo CLIP

In [ ]:
MODEL_ID = "openai/clip-vit-base-patch32"

model = CLIPModel.from_pretrained(MODEL_ID).to(DEVICE).eval()
processor = CLIPProcessor.from_pretrained(MODEL_ID)

print(f"Modelo {MODEL_ID} cargado en {DEVICE}.")
print(f"  Dimensión embedding: {model.config.projection_dim}")

## 3 · Carga de Datos

Cargamos el dataset JSONL y agrupamos cada `group_id` en un par `{real, fake}` con sus respectivos titulares e imágenes.

In [ ]:
# ── Montar Drive (Colab) ─────────────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive")

BASE_DIR   = Path("/content/drive/MyDrive/TFG")
JSONL_PATH = BASE_DIR / "titles_img_data.jsonl"
FAKE_IMG_DIR = BASE_DIR / "dataset" / "fake_images"

print(f"Dataset path:  {JSONL_PATH}")
print(f"Fake images:   {FAKE_IMG_DIR}")

In [ ]:
# ── Cargar y emparejar ───────────────────────────────────────────────
df = pd.read_json(JSONL_PATH, lines=True)
print(f"Total registros: {len(df)}")
print(f"Columnas: {list(df.columns)}")
display(df.head(4))

# Agrupar por group_id → dict[group_id, {"real": row, "fake": row}]
pairs: dict[str, dict] = {}
for _, row in df.iterrows():
    gid = row["group_id"]
    label = "real" if row["is_real"] == 1 else "fake"
    pairs.setdefault(gid, {})[label] = row.to_dict()

# Filtrar grupos incompletos
complete_pairs = {k: v for k, v in pairs.items() if "real" in v and "fake" in v}
print(f"\nPares completos: {len(complete_pairs)} / {len(pairs)} grupos")

## 4 · Utilidades de Carga de Imagen

In [ ]:
def load_image_from_url(url: str, timeout: int = 15) -> Image.Image | None:
    """Descarga una imagen desde URL y la devuelve como PIL Image RGB."""
    try:
        resp = requests.get(url, timeout=timeout, headers={"User-Agent": "Mozilla/5.0"})
        resp.raise_for_status()
        return Image.open(io.BytesIO(resp.content)).convert("RGB")
    except Exception:
        return None


def load_image_from_path(path: str | Path) -> Image.Image | None:
    """Abre una imagen local y la devuelve como PIL Image RGB."""
    try:
        return Image.open(path).convert("RGB")
    except Exception:
        return None


def load_image(img_path: str) -> Image.Image | None:
    """Carga una imagen: detecta automáticamente URL vs. ruta local."""
    if img_path.startswith("http"):
        return load_image_from_url(img_path)
    # Ruta local — resolver relativa a BASE_DIR si no es absoluta
    p = Path(img_path)
    if not p.is_absolute():
        p = BASE_DIR / p
    return load_image_from_path(p)

## 5 · Extracción Dual-Path de Embeddings

Para cada grupo extraemos **cuatro embeddings normalizados**:

| Símbolo | Descripción |
|---|---|
| $E_{Img\_Real}$ | Embedding visual de la imagen real |
| $E_{Img\_Sint}$ | Embedding visual de la imagen sintética (FLUX) |
| $E_{T\_Real}$ | Embedding textual del titular real |
| $E_{T\_Sint}$ | Embedding textual del titular sintético |

In [ ]:
@torch.no_grad()
def encode_image(image: Image.Image) -> torch.Tensor:
    """Devuelve el embedding visual normalizado (1, D) en GPU."""
    inputs = processor(images=image, return_tensors="pt").to(DEVICE)
    emb = model.get_image_features(**inputs)
    return F.normalize(emb, dim=-1)


@torch.no_grad()
def encode_text(text: str) -> torch.Tensor:
    """Devuelve el embedding textual normalizado (1, D) en GPU."""
    inputs = processor(text=[text], return_tensors="pt", truncation=True, max_length=77).to(DEVICE)
    emb = model.get_text_features(**inputs)
    return F.normalize(emb, dim=-1)


def cosine_sim(a: torch.Tensor, b: torch.Tensor) -> float:
    """Similitud coseno escalar entre dos embeddings normalizados."""
    return (a @ b.T).item()

In [ ]:
# ── Procesar todos los pares ─────────────────────────────────────────
results = []
skipped = 0

for gid, pair in tqdm(complete_pairs.items(), desc="Procesando pares"):
    real_row = pair["real"]
    fake_row = pair["fake"]

    # Cargar imágenes
    img_real = load_image(real_row["img_path"])
    img_fake = load_image(fake_row["img_path"])

    if img_real is None or img_fake is None:
        skipped += 1
        continue

    # Embeddings visuales
    e_img_real = encode_image(img_real)
    e_img_fake = encode_image(img_fake)

    # Embeddings textuales
    e_t_real = encode_text(real_row["title"])
    e_t_sint = encode_text(fake_row["title"])

    # ── Enfoque A: Fidelidad Original (Imagen Real) ──────────────────
    sim_A_real = cosine_sim(e_img_real, e_t_real)    # sim(ImgReal, TitReal)
    sim_A_sint = cosine_sim(e_img_real, e_t_sint)    # sim(ImgReal, TitSint)

    # ── Enfoque B: Sesgo de Generación (Imagen Sintética) ────────────
    sim_B_real = cosine_sim(e_img_fake, e_t_real)    # sim(ImgSint, TitReal)
    sim_B_sint = cosine_sim(e_img_fake, e_t_sint)    # sim(ImgSint, TitSint)

    results.append({
        "group_id":     gid,
        "title_real":   real_row["title"],
        "title_sint":   fake_row["title"],
        # Enfoque A
        "sim_A_real":   sim_A_real,
        "sim_A_sint":   sim_A_sint,
        "delta_A":      sim_A_real - sim_A_sint,
        # Enfoque B
        "sim_B_real":   sim_B_real,
        "sim_B_sint":   sim_B_sint,
        "delta_B":      sim_B_sint - sim_B_real,
    })

print(f"\n✅ Procesados: {len(results)} | ⏭️ Omitidos (imagen no cargada): {skipped}")

In [ ]:
# ── Convertir a DataFrame ────────────────────────────────────────────
res_df = pd.DataFrame(results)
display(res_df.describe().round(4))
res_df.head()

## 6 · Enfoque A — Fidelidad Original

**Pregunta:** Dada la **imagen real**, ¿cuál titular se alinea semánticamente mejor?

$$\text{Predicción A} = \arg\max\bigl(\cos(E_{Img\_Real},\, E_{T\_Real}),\;\cos(E_{Img\_Real},\, E_{T\_Sint})\bigr)$$

- Si $\Delta_A = \cos_{real} - \cos_{sint} > 0$ → el titular **real** tiene mayor afinidad.
- Si $\Delta_A < 0$ → el parafraseo ha *diluido* el significado o incluso se alinea mejor por generalización.

In [ ]:
# ── Estadísticas del Enfoque A ───────────────────────────────────────
wins_A_real = (res_df["delta_A"] > 0).sum()
wins_A_sint = (res_df["delta_A"] < 0).sum()
ties_A      = (res_df["delta_A"] == 0).sum()
total = len(res_df)

print("═" * 55)
print("  ENFOQUE A — FIDELIDAD ORIGINAL (Imagen Real)")
print("═" * 55)
print(f"  Titular Real gana:      {wins_A_real:>4d}  ({wins_A_real/total:.1%})")
print(f"  Titular Sintético gana: {wins_A_sint:>4d}  ({wins_A_sint/total:.1%})")
print(f"  Empates:                {ties_A:>4d}  ({ties_A/total:.1%})")
print("─" * 55)
print(f"  Δ_A medio:  {res_df['delta_A'].mean():+.4f}")
print(f"  Δ_A mediana:{res_df['delta_A'].median():+.4f}")
print(f"  Δ_A std:    {res_df['delta_A'].std():.4f}")
print("═" * 55)

## 7 · Enfoque B — Sesgo de Generación (*Prompt-Adherence Bias*)

**Pregunta:** Dada la **imagen sintética** (generada por FLUX a partir del titular sintético), ¿cuál titular se alinea mejor?

$$\text{Predicción B} = \arg\max\bigl(\cos(E_{Img\_Sint},\, E_{T\_Real}),\;\cos(E_{Img\_Sint},\, E_{T\_Sint})\bigr)$$

- Si $\Delta_B = \cos_{sint} - \cos_{real} > 0$ → **Prompt-Adherence Bias** confirmado: la imagen generada se adhiere más a su prompt de origen.
- Una alta correlación en este enfoque es un **indicio de contenido generado**, no de calidad periodística.

In [ ]:
# ── Estadísticas del Enfoque B ───────────────────────────────────────
wins_B_sint = (res_df["delta_B"] > 0).sum()
wins_B_real = (res_df["delta_B"] < 0).sum()
ties_B      = (res_df["delta_B"] == 0).sum()

print("═" * 55)
print("  ENFOQUE B — SESGO DE GENERACIÓN (Imagen Sintética)")
print("═" * 55)
print(f"  Titular Sintético gana: {wins_B_sint:>4d}  ({wins_B_sint/total:.1%})")
print(f"  Titular Real gana:      {wins_B_real:>4d}  ({wins_B_real/total:.1%})")
print(f"  Empates:                {ties_B:>4d}  ({ties_B/total:.1%})")
print("─" * 55)
print(f"  Δ_B medio:  {res_df['delta_B'].mean():+.4f}")
print(f"  Δ_B mediana:{res_df['delta_B'].median():+.4f}")
print(f"  Δ_B std:    {res_df['delta_B'].std():.4f}")
print("═" * 55)

## 8 · Sistema de Decisión y Matriz de Confusión

### Regla de Clasificación (basada en Enfoque A)

Para cada par, clasificamos como **"Real"** el titular con mayor similitud coseno frente a la **imagen real**:

$$\hat{y} = \begin{cases} \text{Real} & \text{si } \cos(E_{Img\_Real}, E_{T\_Real}) \geq \cos(E_{Img\_Real}, E_{T\_Sint}) \\ \text{Sintético} & \text{en caso contrario} \end{cases}$$

Documentamos los **casos de confusión**: pares donde el Enfoque B muestra alta adherencia al prompt ($\Delta_B \gg 0$) pero el Enfoque A clasifica incorrectamente.

In [ ]:
# ── Etiquetas de clasificación ───────────────────────────────────────
# Para cada grupo generamos DOS filas: una predicción para el titular real y otra para el sintético.
# Ground truth: 1 = Real, 0 = Sintético
# Predicción: si delta_A > 0 → predice correctamente que Real > Sintético

y_true = []  # etiqueta real de cada titular
y_pred = []  # predicción basada en Enfoque A

for _, row in res_df.iterrows():
    # Para el titular real: ¿el sistema lo identifica como el "ganador"?
    # delta_A > 0 significa sim(Img_Real, T_Real) > sim(Img_Real, T_Sint)
    # → predice correctamente que el real es "Real"
    pred_real_is_winner = row["delta_A"] >= 0

    # Titular Real
    y_true.append(1)  # es real
    y_pred.append(1 if pred_real_is_winner else 0)

    # Titular Sintético
    y_true.append(0)  # es sintético
    y_pred.append(0 if pred_real_is_winner else 1)

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print("═" * 55)
print("  SISTEMA DE DECISIÓN — Enfoque A")
print("═" * 55)
print(f"  Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
print()
print(classification_report(y_true, y_pred, target_names=["Sintético", "Real"]))

In [ ]:
# ── Casos de confusión: el Enfoque B tiene alta adherencia pero A falla
confusion_cases = res_df[
    (res_df["delta_A"] < 0) &   # Enfoque A se equivoca (sint gana sobre real)
    (res_df["delta_B"] > 0)     # Enfoque B muestra alto prompt-adherence
].copy()

print(f"Casos de confusión (A falla + B muestra Prompt-Adherence): {len(confusion_cases)}")
print(f"Porcentaje sobre el total: {len(confusion_cases)/total:.1%}")
print()

if len(confusion_cases) > 0:
    print("Top 10 casos con mayor |Δ_A| (mayor error del Enfoque A):")
    display(
        confusion_cases
        .nsmallest(10, "delta_A")
        [["group_id", "title_real", "title_sint",
          "sim_A_real", "sim_A_sint", "delta_A",
          "sim_B_real", "sim_B_sint", "delta_B"]]
    )

## 9 · Visualizaciones

In [ ]:
# ── Configuración de estilo ──────────────────────────────────────────
COLOR_REAL = "#2196F3"    # Azul
COLOR_SINT = "#FF5722"    # Naranja-rojo
COLOR_DELTA_POS = "#4CAF50"  # Verde
COLOR_DELTA_NEG = "#F44336"  # Rojo
ALPHA = 0.55
BINS = 50

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Fig 1 — Distribuciones de Similitud: Enfoque A vs Enfoque B
# ═══════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5), sharey=True)

# ── Panel izquierdo: Enfoque A ───────────────────────────────────────
ax = axes[0]
ax.hist(res_df["sim_A_real"], bins=BINS, alpha=ALPHA, color=COLOR_REAL,
        label=r"$\cos(Img_{Real},\, T_{Real})$", edgecolor="white", linewidth=0.5)
ax.hist(res_df["sim_A_sint"], bins=BINS, alpha=ALPHA, color=COLOR_SINT,
        label=r"$\cos(Img_{Real},\, T_{Sint})$", edgecolor="white", linewidth=0.5)
ax.axvline(res_df["sim_A_real"].mean(), color=COLOR_REAL, ls="--", lw=1.5,
           label=f'Media Real = {res_df["sim_A_real"].mean():.3f}')
ax.axvline(res_df["sim_A_sint"].mean(), color=COLOR_SINT, ls="--", lw=1.5,
           label=f'Media Sint = {res_df["sim_A_sint"].mean():.3f}')
ax.set_title("Enfoque A — Fidelidad Original\n(Similitud con Imagen Real)",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Similitud Coseno")
ax.set_ylabel("Frecuencia")
ax.legend(fontsize=9, loc="upper left")

# ── Panel derecho: Enfoque B ─────────────────────────────────────────
ax = axes[1]
ax.hist(res_df["sim_B_real"], bins=BINS, alpha=ALPHA, color=COLOR_REAL,
        label=r"$\cos(Img_{Sint},\, T_{Real})$", edgecolor="white", linewidth=0.5)
ax.hist(res_df["sim_B_sint"], bins=BINS, alpha=ALPHA, color=COLOR_SINT,
        label=r"$\cos(Img_{Sint},\, T_{Sint})$", edgecolor="white", linewidth=0.5)
ax.axvline(res_df["sim_B_real"].mean(), color=COLOR_REAL, ls="--", lw=1.5,
           label=f'Media Real = {res_df["sim_B_real"].mean():.3f}')
ax.axvline(res_df["sim_B_sint"].mean(), color=COLOR_SINT, ls="--", lw=1.5,
           label=f'Media Sint = {res_df["sim_B_sint"].mean():.3f}')
ax.set_title("Enfoque B — Sesgo de Generación\n(Similitud con Imagen Sintética)",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Similitud Coseno")
ax.legend(fontsize=9, loc="upper left")

plt.tight_layout()
plt.savefig("fig_similarity_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Fig 2 — Distribuciones de Δ (Delta) para cada Enfoque
# ═══════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5), sharey=True)

# ── Delta A ──────────────────────────────────────────────────────────
ax = axes[0]
colors_A = [COLOR_DELTA_POS if d >= 0 else COLOR_DELTA_NEG for d in res_df["delta_A"]]
n, bin_edges, patches = ax.hist(res_df["delta_A"], bins=BINS, edgecolor="white", linewidth=0.5)
# Colorear barras según signo
for patch, left_edge in zip(patches, bin_edges):
    patch.set_facecolor(COLOR_DELTA_POS if left_edge >= 0 else COLOR_DELTA_NEG)
    patch.set_alpha(0.7)
ax.axvline(0, color="black", ls="-", lw=1)
ax.axvline(res_df["delta_A"].mean(), color="navy", ls="--", lw=1.5,
           label=f'Media Δ_A = {res_df["delta_A"].mean():+.4f}')
ax.set_title(r"$\Delta_A = \cos_{Real} - \cos_{Sint}$" + "\n(Imagen Real)",
             fontsize=13, fontweight="bold")
ax.set_xlabel(r"$\Delta_A$")
ax.set_ylabel("Frecuencia")
ax.legend(fontsize=10)

# ── Delta B ──────────────────────────────────────────────────────────
ax = axes[1]
n, bin_edges, patches = ax.hist(res_df["delta_B"], bins=BINS, edgecolor="white", linewidth=0.5)
for patch, left_edge in zip(patches, bin_edges):
    patch.set_facecolor(COLOR_DELTA_POS if left_edge >= 0 else COLOR_DELTA_NEG)
    patch.set_alpha(0.7)
ax.axvline(0, color="black", ls="-", lw=1)
ax.axvline(res_df["delta_B"].mean(), color="navy", ls="--", lw=1.5,
           label=f'Media Δ_B = {res_df["delta_B"].mean():+.4f}')
ax.set_title(r"$\Delta_B = \cos_{Sint} - \cos_{Real}$" + "\n(Imagen Sintética)",
             fontsize=13, fontweight="bold")
ax.set_xlabel(r"$\Delta_B$")
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig("fig_delta_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Fig 3 — Matriz de Confusión
# ═══════════════════════════════════════════════════════════════════════
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Absoluta
disp_abs = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Sintético", "Real"]
)
disp_abs.plot(ax=axes[0], cmap="Blues", values_format="d", colorbar=False)
axes[0].set_title("Matriz de Confusión (absoluta)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Predicción")
axes[0].set_ylabel("Etiqueta Real")

# Normalizada
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
disp_norm = ConfusionMatrixDisplay(
    confusion_matrix=cm_norm,
    display_labels=["Sintético", "Real"]
)
disp_norm.plot(ax=axes[1], cmap="Oranges", values_format=".2%", colorbar=False)
axes[1].set_title("Matriz de Confusión (normalizada)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Predicción")
axes[1].set_ylabel("Etiqueta Real")

plt.tight_layout()
plt.savefig("fig_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Fig 4 — Scatter: Δ_A vs Δ_B (cada punto = un grupo)
# ═══════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(8, 7))

scatter = ax.scatter(
    res_df["delta_A"], res_df["delta_B"],
    c=res_df["delta_A"].apply(lambda x: COLOR_DELTA_POS if x >= 0 else COLOR_DELTA_NEG),
    alpha=0.45, s=25, edgecolors="white", linewidths=0.3
)

ax.axhline(0, color="gray", ls="--", lw=0.8)
ax.axvline(0, color="gray", ls="--", lw=0.8)

# Anotar cuadrantes
ax.text(0.05, 0.95, "Fidelidad Alta\nAlt. Adherence",
        transform=ax.transAxes, fontsize=9, va="top", color="#388E3C", fontweight="bold")
ax.text(0.70, 0.95, "Fidelidad Baja\nAlto Adherence",
        transform=ax.transAxes, fontsize=9, va="top", color="#D32F2F", fontweight="bold")
ax.text(0.05, 0.05, "Fidelidad Alta\nBajo Adherence",
        transform=ax.transAxes, fontsize=9, va="bottom", color="#1976D2", fontweight="bold")
ax.text(0.70, 0.05, "Fidelidad Baja\nBajo Adherence",
        transform=ax.transAxes, fontsize=9, va="bottom", color="#757575", fontweight="bold")

ax.set_xlabel(r"$\Delta_A$ (Fidelidad Original)", fontsize=12)
ax.set_ylabel(r"$\Delta_B$ (Sesgo de Generación)", fontsize=12)
ax.set_title(r"Relación entre $\Delta_A$ y $\Delta_B$ por grupo",
             fontsize=14, fontweight="bold")

plt.tight_layout()
plt.savefig("fig_scatter_deltas.png", dpi=150, bbox_inches="tight")
plt.show()

## 10 · Resumen de Resultados

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Tabla resumen
# ═══════════════════════════════════════════════════════════════════════
summary = pd.DataFrame({
    "Métrica": [
        "Pares evaluados",
        "Pares omitidos",
        "──────────────────────────────────",
        "[A] Win-Rate Titular Real",
        "[A] Δ_A medio",
        "[A] Δ_A mediana",
        "──────────────────────────────────",
        "[B] Win-Rate Titular Sintético",
        "[B] Δ_B medio",
        "[B] Δ_B mediana",
        "──────────────────────────────────",
        "Accuracy del clasificador (Enfoque A)",
        "Casos de confusión (A falla + B alta adherencia)",
    ],
    "Valor": [
        f"{total}",
        f"{skipped}",
        "",
        f"{wins_A_real/total:.2%}",
        f"{res_df['delta_A'].mean():+.4f}",
        f"{res_df['delta_A'].median():+.4f}",
        "",
        f"{wins_B_sint/total:.2%}",
        f"{res_df['delta_B'].mean():+.4f}",
        f"{res_df['delta_B'].median():+.4f}",
        "",
        f"{accuracy_score(y_true, y_pred):.4f}",
        f"{len(confusion_cases)} ({len(confusion_cases)/total:.1%})",
    ]
})
display(summary.style.hide(axis="index").set_properties(**{"text-align": "left"}))

## 11 · Conclusiones

### Enfoque A — Fidelidad Original
- Evalúa si el titular **real** mantiene mayor coherencia semántica con su imagen original que la paráfrasis sintética.
- Un $\Delta_A > 0$ mayoritario confirma que el parafraseo introduce una **dilución semántica** detectable por CLIP.

### Enfoque B — Sesgo de Generación  
- Demuestra el **Prompt-Adherence Bias**: las imágenes generadas por FLUX se alinean de forma casi perfecta con el titular que usaron como prompt.
- Una correlación excesivamente alta en este enfoque actúa como **indicador de contenido generado artificialmente**.

### Implicaciones para la Detección
- La similitud coseno CLIP, sin entrenamiento supervisado, puede servir como **feature** discriminativa en un clasificador multimodal.
- Los "casos de confusión" revelan las limitaciones: cuando la paráfrasis es más descriptiva que el titular original, CLIP puede preferirla.